# 04 · Evaluate the claim—not just the demonstration

The release compares two non-neural search orders, three graph-model seeds, three MLP seeds, and three provenance-ablation seeds. The held-out unit is an entire scene, never individual near-duplicate candidate patches.

Results were measured on a small synthetic benchmark. OOD changes several factors together; it does not isolate the causal effect of each factor. This is not a user study of believable characters.

In [ ]:
from pathlib import Path
import sys, json, os
ROOT = Path.cwd().resolve()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if not (ROOT / "src" / "belieftown").exists():
    raise RuntimeError("Run this notebook from the source repository or its notebooks directory.")
sys.path.insert(0, str(ROOT / "src"))
RUN_ROOT = Path(os.environ["BELIEFTOWN_RUN_ROOT"]).resolve() if os.environ.get("BELIEFTOWN_RUN_ROOT") else None
DATA = RUN_ROOT / "data" if RUN_ROOT else ROOT / "benchmarks" / "data"
CHECKPOINTS = RUN_ROOT / "checkpoints" if RUN_ROOT else ROOT / "artifacts" / "checkpoints"
REPORT = RUN_ROOT / "evaluation" if RUN_ROOT else ROOT / "artifacts" / "evaluation"
AUDIT_DIR = RUN_ROOT if RUN_ROOT else ROOT / "artifacts"
OUTPUT = AUDIT_DIR / "notebook_run"
OUTPUT.mkdir(parents=True, exist_ok=True)
import numpy as np
get_ipython().run_line_magic("matplotlib", "inline")
import matplotlib.pyplot as plt
from IPython.display import display, HTML

In [ ]:
import csv, hashlib
results = json.loads((REPORT / "results.json").read_text())
with (REPORT / "search_results.csv").open() as f:
    rows = list(csv.DictReader(f))
assert results["all_searches_correct"]
assert all(int(row["correct"]) == 1 for row in rows)
assert all(float(row["minimum_gap"]) == 0 for row in rows if row["minimum_gap"] != "")
print("Search runs:", len(rows))
print("All recorded feasibility / bounded-minimum outcomes agree with exhaustive labels.")
print("Statistical unit:", results["protocol"]["split_unit"])
print("Training seeds in this report:", results["protocol"]["training_seeds"])
print("Report directory:", REPORT)

## Matched comparisons and uncertainty

The analysis averages a method’s three training-seed outcomes **within each scene**, then bootstraps scene means for a 95% interval. It does not treat every candidate patch as an independent data point. Paired differences use the same scenes and the heuristic as the reference.

Verifier-call budgets are not wall-clock deadlines. End-to-end request latency includes graph construction and model scoring but excludes amortized model loading and scene construction.

In [ ]:
import html
lines = ["<table><tr><th>Split</th><th>Method</th><th>Mean calls</th><th>Mean latency (ms)</th><th>95% latency interval</th></tr>"]
for item in results["summary"]:
    latency = item["latency_ms"]
    lines.append(f"<tr><td>{html.escape(item['split'])}</td><td>{html.escape(item['method'])}</td><td>{item['verifier_calls']['mean']:.2f}</td><td>{latency['mean']:.2f}</td><td>[{latency['low']:.2f}, {latency['high']:.2f}]</td></tr>")
lines.append("</table>")
display(HTML("".join(lines)))

In [ ]:
test = [x for x in results["summary"] if x["split"] == "test"]
plt.figure(figsize=(9, 4))
plt.bar([x["method"] for x in test], [x["latency_ms"]["mean"] for x in test])
plt.ylabel("Mean steady-state search latency (ms)")
plt.xticks(rotation=18, ha="right")
plt.title("Measured candidate-ordering latency, including neural overhead")
plt.tight_layout()
plt.show()

## Belief quality is a separate measurement

For an actual room $X$, the multiclass Brier score is

$$\mathrm{BS}=\frac1M\sum_n\sum_x\left(b_n(x)-\mathbb 1[X_n=x]\right)^2.$$

The belief comparison uses the **same initial received histories**, not divergent trajectories. Generated subjective priors vary across NPCs. The score is not a measure of human-likeness or a proof that a sensor model matches reality.

In [ ]:
for item in results["belief_summary"]:
    print(item["split"], item["mode"], "mean Brier:", round(item["brier"]["mean"], 5))
# Validate data provenance before interpreting any learned result.
manifest_path = DATA / "manifest.json"
manifest = json.loads(manifest_path.read_text())
assert hashlib.sha256(manifest_path.read_bytes()).hexdigest() == results["dataset_manifest_sha256"]
for split, entry in manifest["splits"].items():
    path = DATA / f"{split}.jsonl"
    assert hashlib.sha256(path.read_bytes()).hexdigest() == entry["sha256"]
for name, expected in results["checkpoint_hashes"].items():
    path = CHECKPOINTS / f"{name}.pt"
    assert hashlib.sha256(path.read_bytes()).hexdigest() == expected
print("Dataset and checkpoint hashes: verified")

## What the experiment supports

The exact search and replay workflow produces auditable results in its finite model. Learned ordering is implemented and measurable, but it is not faster than the heuristic on this release’s small scenes. Infeasible goals still require exhaustive bounded search for a no-solution certificate.

A responsible launch emphasizes the **inspect → intervene → verify → replay** workflow, publishes the negative result, and states the assumptions. The README and model/data cards do that. Rebuilding the experiment can change measured latency; update the report rather than reusing an old timing claim.